In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd "/content/drive/MyDrive/Colab Data/BI and ML"

/content/drive/MyDrive/Colab Data/BI and ML


In [3]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')

In [7]:
# 2. INITIAL EXPLORATION & CLEANING
# Initial data inspection
print("=== Initial Data Summary ===")
print(f"Records: {len(df)}")
print(f"Columns: {len(df.columns)}")
print("\nMissing values per column:")
print(df.isnull().sum())

=== Initial Data Summary ===
Records: 180519
Columns: 53

Missing values per column:
Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                           

In [8]:
# Remove unnecessary columns
cols_to_drop = [
    'Product Description', 'Product Image', 'Customer Password', 'Customer Email',
    'Customer Fname', 'Customer Lname', 'Order Zipcode', 'Latitude', 'Longitude'
]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

In [9]:
# Remove duplicate helper columns after validation
# Ensure columns are identical
if (df['Customer Id'] == df['Order Customer Id']).all():
    df = df.drop(columns=['Order Customer Id'])
if (df['Product Card Id'] == df['Order Item Cardprod Id']).all():
    df = df.drop(columns=['Order Item Cardprod Id'])

# Remove spaces from column names and fix common issues
df.columns = df.columns.str.replace(' ', '')
df = df.rename(columns={
    'Daysforshipping(real)': 'DaysForShippingReal','Daysforshipment(scheduled)' : 'DaysForShipmentScheduled'
})

In [8]:
# 3. HANDLE MISSING VALUES
# -----------------------------

# Fill missing CustomerZipcode with string "0.0"
df['CustomerZipcode'] = df['CustomerZipcode'].fillna('0.0')

In [9]:
df.columns

Index(['Type', 'DaysForShippingReal', 'DaysForShipmentScheduled',
       'Benefitperorder', 'Salespercustomer', 'DeliveryStatus',
       'Late_delivery_risk', 'CategoryId', 'CategoryName', 'CustomerCity',
       'CustomerCountry', 'CustomerId', 'CustomerSegment', 'CustomerState',
       'CustomerStreet', 'CustomerZipcode', 'DepartmentId', 'DepartmentName',
       'Market', 'OrderCity', 'OrderCountry', 'orderdate(DateOrders)',
       'OrderId', 'OrderItemDiscount', 'OrderItemDiscountRate', 'OrderItemId',
       'OrderItemProductPrice', 'OrderItemProfitRatio', 'OrderItemQuantity',
       'Sales', 'OrderItemTotal', 'OrderProfitPerOrder', 'OrderRegion',
       'OrderState', 'OrderStatus', 'ProductCardId', 'ProductCategoryId',
       'ProductName', 'ProductPrice', 'ProductStatus',
       'shippingdate(DateOrders)', 'ShippingMode'],
      dtype='object')

In [10]:
# change to datetime format
df['orderdate(DateOrders)'] = pd.to_datetime(df['orderdate(DateOrders)'])
df['shippingdate(DateOrders)'] = pd.to_datetime(df['shippingdate(DateOrders)'])

In [11]:
# 5. ROUND NUMERIC COLUMNS
# -----------------------------

two_decimals = [
    'Benefitperorder', 'OrderItemDiscount', 'OrderItemProductPrice',
    'Sales', 'OrderItemTotal', 'OrderProfitPerOrder', 'ProductPrice'
]
six_decimals = [
    'OrderItemDiscountRate', 'OrderItemProfitRatio',
]

# Round
df[two_decimals] = df[two_decimals].round(2)
df[six_decimals] = df[six_decimals].round(6)

In [12]:
# -----------------------------
# 6. CREATE DIMENSION TABLES
# -----------------------------

# -----------------
# DimCustomer
# -----------------
dim_customer = df[[
    'CustomerId', 'CustomerSegment', 'CustomerCountry', 'CustomerState',
    'CustomerCity', 'CustomerStreet', 'CustomerZipcode'
]].drop_duplicates().reset_index(drop=True)

# -----------------
# DimProduct
# -----------------
dim_product = df[[
    'ProductCardId', 'ProductCategoryId', 'CategoryName',
    'ProductName', 'ProductPrice', 'ProductStatus'
]].drop_duplicates().reset_index(drop=True)

# -----------------
# DimDepartment
# -----------------
dim_department = df[['DepartmentName', 'Market']].drop_duplicates().reset_index(drop=True)


# -----------------
# DimLocation
# -----------------
dim_location = df[['OrderRegion', 'OrderCity', 'OrderState', 'OrderCountry']].drop_duplicates().reset_index(drop=True)

# -----------------
# DimShipping
# -----------------
dim_shipping = df[['ShippingMode', 'DaysForShippingReal', 'DaysForShipmentScheduled']].drop_duplicates().reset_index(drop=True)


# -----------------
# DimOrderStatus
# -----------------
dim_orderstatus = df[['OrderStatus']].drop_duplicates().reset_index(drop=True)

# -----------------
# DimTransactionType
# -----------------
dim_transactiontype = df[['Type']].drop_duplicates().reset_index(drop=True)

# -----------------
# DimDate
# -----------------
# Compile all unique dates from order and shipping for the date dimension
all_dates = pd.concat([
    pd.to_datetime(df['orderdate(DateOrders)']).dt.date,
    pd.to_datetime(df['shippingdate(DateOrders)']).dt.date
]).dropna().unique()

dim_date = pd.DataFrame({'DateId': pd.to_datetime(all_dates)})
dim_date = dim_date.drop_duplicates().sort_values('DateId').reset_index(drop=True)
dim_date['Year'] = dim_date['DateId'].dt.year
dim_date['Quarter'] = dim_date['DateId'].dt.quarter
dim_date['Month'] = dim_date['DateId'].dt.month
dim_date['MonthName'] = dim_date['DateId'].dt.strftime('%B')
dim_date['Day'] = dim_date['DateId'].dt.day
dim_date['DayOfWeek'] = dim_date['DateId'].dt.weekday + 1
dim_date['DayName'] = dim_date['DateId'].dt.strftime('%A')

# -----------------------------
# PREVIEW THE TABLES
# -----------------------------
print("Customer dimension:\n", dim_customer.head())
print("Product dimension:\n", dim_product.head())
print("Department dimension:\n", dim_department.head())
print("Location dimension:\n", dim_location.head())
print("Shipping dimension:\n", dim_shipping.head())
print("Order status dimension:\n", dim_orderstatus.head())
print("Transaction type dimension:\n", dim_transactiontype.head())
print("Date dimension:\n", dim_date.head())

# -----------------------------
# EXPORT TO CSV
# -----------------------------
dim_customer.to_csv('DimCustomer.csv', index=False)
dim_product.to_csv('DimProduct.csv', index=False)
dim_department.to_csv('DimDepartment.csv', index=False)
dim_location.to_csv('DimLocation.csv', index=False)
dim_shipping.to_csv('DimShipping.csv', index=False)
dim_orderstatus.to_csv('DimOrderStatus.csv', index=False)
dim_transactiontype.to_csv('DimTransactionType.csv', index=False)
dim_date.to_csv('DimDate.csv', index=False)

print('All dimension tables created and exported to CSV.')

Customer dimension:
    CustomerId CustomerSegment CustomerCountry CustomerState CustomerCity  \
0       20755        Consumer     Puerto Rico            PR       Caguas   
1       19492        Consumer     Puerto Rico            PR       Caguas   
2       19491        Consumer         EE. UU.            CA     San Jose   
3       19490     Home Office         EE. UU.            CA  Los Angeles   
4       19489       Corporate     Puerto Rico            PR       Caguas   

             CustomerStreet CustomerZipcode  
0  5365 Noble Nectar Island           725.0  
1          2679 Rustic Loop           725.0  
2      8510 Round Bear Gate         95125.0  
3           3200 Amber Bend         90027.0  
4  8671 Iron Anchor Corners           725.0  
Product dimension:
    ProductCardId  ProductCategoryId     CategoryName  \
0           1360                 73   Sporting Goods   
1            365                 17           Cleats   
2            627                 29    Shop By Sport   
3 

In [13]:
# -----------------------------
# CREATE FACT TABLE: FactOrderItem
# -----------------------------

fact_orderitem = df[[
    'OrderId',
    'OrderItemId',
    'CustomerId',
    'ProductCardId',
    'DepartmentName', 'Market',
    'OrderCity', 'OrderState', 'OrderCountry',  # as natural keys for location
    'ShippingMode', 'DaysForShippingReal', 'DaysForShipmentScheduled',    # as natural keys for shipping
    'OrderStatus',                   # as natural key for order status
    'Type',                          # as natural key for transaction type
    'orderdate(DateOrders)',         # as natural key for date
    'shippingdate(DateOrders)',      # as natural key for date
    'OrderItemQuantity',
    'OrderItemDiscount',
    'OrderItemDiscountRate',
    'OrderItemProductPrice',
    'OrderItemProfitRatio',
    'Sales',
    'OrderItemTotal',
    'Benefitperorder',
    'OrderProfitPerOrder',
    'DeliveryStatus',
    'Late_delivery_risk'
]].drop_duplicates().reset_index(drop=True)
fact_orderitem.to_csv('FactOrderItem.csv', index=False)